# 06 SPY・TLT・GLDの資産配分比較

[研究案内](../資料/07_研究コードと成果物の案内.md) / [実装元](../src/quant_ai/risk_allocation.py)。VS Codeで関数名をF12、または下のセルで実装を表示。

## 1. 目的と仮説
過去リスクだけの3配分を等ウェイトと比較する。

## 2. 理論と数式
## 数式：どの情報で重みを決めるか

月末tまでの直近L=252個の日次単純リターン $r_{i,s}=P_{i,s}/P_{i,s-1}-1$ を使います。

1. 等ウェイト：$w_i=1/3$。各月この比率に戻す。
2. 逆ボラ：$w_i=(1/\hat\sigma_i)/\sum_j(1/\hat\sigma_j)$。相関は使わない。
3. 最小分散：$\min_w w^\top\hat\Sigma_t w$, $\sum_iw_i=1$, $w_i\ge0$。
4. 縮小最小分散：Ledoit-Wolf共分散を使う参考条件。標本共分散の推定誤差に対処する。

月末終値が確定してから判断し、翌共通取引日終値で約定します。
新保有の最初の価格損益はさらにその次の日です。翌日の値動きを先取りしません。
月中は価格変化で保有比率がドリフトし、毎日目標比率に戻すことはしません。
`audits`で履歴終端・判断・約定・損益開始を照合できます。

## 指標の定義
幾何年率CAGR $=(\prod_t(1+r_t))^{252/n}-1$、年率変動率 $=\sqrt{252}\,s(r)$。
ここでSharpeは無リスク金利を0と置いた比較用指標 $\sqrt{252}\,\bar r/s(r)$ です。
実際の短期金利を引いたSharpeではありません。
最大DDは初期資金も含む過去ピークからの最大下落率で、マイナスが大きいほど悪化します。
turnoverは年間の売買金額/NAVで、初回購入・売却と購入の両側を含みます。

全戦略で同じ評価期間・執行・コスト条件です。初回の評価日は現金からの買付コストのみが
発生するため、初日から最適比率で保有していたことにはしません。

## 3. データの定義
## データの意味と制約
Yahoo Financeをyfinanceで取得した配当・分割調整済み終値を使います。
最新日が欠損なら取得した最終有効日までとし、その日付を上の表で確認します。
欠損行は3ETF共通の観測日にそろえるため明示的に除外し、値を補完しません。
ETF経費は価格に含まれるので再控除しません。分配金再投資相当の理想化であり、
税・再投資タイミング・実際の約定価格との差は残ります。円換算の投資成果ではありません。

[SPY公式](https://www.ssga.com/us/en/individual/etfs/state-street-spdr-sp-500-etf-trust-spy)、
[TLT公式](https://www.ishares.com/us/products/239454/ishares-20-year-treasury-bond-etf)、
[GLD公式](https://www.spdrgoldshares.com/usa/gld/)。
TLTは金利変動に敏感な長期債ETFで、無リスク資産ではありません。

## 4. 分析条件・パラメータ
2006-01-03〜2026-10-29、252日推定、月末判断・翌終値約定、片道5bps、USD。

## 5. 実装と実行
共通関数の実装も確認してから計算する。

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from quant_ai.risk_allocation import run_allocation_study, period_comparison
from quant_ai.metrics import drawdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/quant_ai").exists())
os.chdir(ROOT)
report_dir = ROOT / "分析結果/レポート/資産配分比較"
data_path = ROOT / "data/processed/spy_tlt_gld_adjusted.parquet"
if not data_path.exists():
    raise FileNotFoundError("先に python scripts/fetch_allocation_data.py を実行してください")
prices = pd.read_parquet(data_path)
metadata = json.loads(data_path.with_suffix(".json").read_text(encoding="utf-8"))
display(pd.Series({"source": metadata["provenance"]["source"],
                   "retrieved_utc": metadata["provenance"]["retrieved_at_utc"],
                   "start": metadata["start"], "end": metadata["end"],
                   "rows": metadata["rows"], "dropped_dates": metadata["dropped_dates"]}))
display(prices.head())


In [ ]:
import inspect
from IPython.display import Code
from quant_ai.risk_allocation import run_allocation_study
print(inspect.getsourcefile(run_allocation_study))
display(Code(inspect.getsource(run_allocation_study), language='python'))

In [ ]:
study = run_allocation_study(prices, lookback=252,
                             evaluation_start="2006-01-01", cost_bps=5)
metrics = study.summary[["annual_return", "volatility", "sharpe",
                          "max_drawdown", "annual_turnover"]]
display(metrics.style.format({"annual_return": "{:.2%}", "volatility": "{:.2%}",
                               "sharpe": "{:.3f}", "max_drawdown": "{:.2%}",
                               "annual_turnover": "{:.2f}"}))
display(study.summary[["start", "end", "observations"]])
display(study.audits["min_variance"].head())


## 6. 計算結果と可視化

In [ ]:
labels = {"equal_weight": "Equal weight", "inverse_vol": "Inverse volatility",
          "min_variance": "Minimum variance", "shrink_min_variance": "Shrinkage min variance"}
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
for method, report in study.reports.items():
    axes[0].plot(report.index, report.wealth, label=labels[method])
    axes[1].plot(report.index, drawdown(report.net_return)*100, label=labels[method])
axes[0].set(yscale="log", ylabel="Wealth (initial=1)", title="Walk-forward, after trading costs")
axes[1].set(ylabel="Drawdown (%)")
for ax in axes:
    ax.grid(alpha=.3)
    ax.legend()
plt.tight_layout()
plt.show()

display(pd.DataFrame({method: targets.mean() for method, targets in study.targets.items()})
        .style.format("{:.1%}"))


## 期間別に結論は変わるか
以下は事前に決めた固定区間です。開始点を変えた区間内ドローダウンなので、全期間のDDと
一致するとは限りません。各区間のannual_returnはその区間の幾何年率です。
たまたま良かった区間だけを抜き出して採用しないでください。

In [ ]:
subperiods = period_comparison(study)
display(subperiods[["annual_return", "volatility", "sharpe", "max_drawdown"]]
        .style.format({"annual_return": "{:.2%}", "volatility": "{:.2%}",
                       "sharpe": "{:.3f}", "max_drawdown": "{:.2%}"}))


## 推定誤差・コスト・結論の不確実性
63/126/252/504日という窓の感度は全条件で2007年開始にそろえます。
主分析の2006年開始と数値を直接混同しないでください。
コスト感度0/5/20bpsも確認し、良かった設定を事後採用しない方針です。

ブロック再標本化は、戦略と等ウェイトを同じ21営業日ブロックで2000回取り直した
**条件付きの記述的区間**です。価格を作り直して重みを再推定したものではなく、
非定常性も十分に反映しないため、一般的な優位性の証明には使いません。
差の区間が0をまたぐ場合は、優位性の主張を特に控えます。

In [ ]:
needed = ["lookback_sensitivity.csv", "cost_sensitivity.csv", "paired_block_intervals.csv"]
if all((report_dir / name).exists() for name in needed):
    lookbacks = pd.read_csv(report_dir / needed[0])
    costs = pd.read_csv(report_dir / needed[1])
    intervals = pd.read_csv(report_dir / needed[2])
    display(lookbacks.pivot(index="lookback", columns="method", values="annual_return"))
    display(lookbacks.pivot(index="lookback", columns="method", values="volatility"))
    display(costs.pivot(index="cost_bps", columns="method", values="annual_return"))
    display(intervals)
else:
    print("感度分析の全出力を作るには python scripts/run_allocation_study.py を実行")


In [ ]:
baseline = study.summary.loc["equal_weight"]
minimum = study.summary.loc["min_variance"]
display(pd.Series({
    "最小分散−等ウェイト CAGR差（年率ポイント）":
        (minimum.annual_return-baseline.annual_return)*100,
    "最小分散−等ウェイト 変動率差（年率ポイント）":
        (minimum.volatility-baseline.volatility)*100,
    "最小分散−等ウェイト 最大DD差（ポイント、正が改善）":
        (minimum.max_drawdown-baseline.max_drawdown)*100,
    "最小分散−等ウェイト Sharpe差（rf=0）": minimum.sharpe-baseline.sharpe,
}))


## 7. 統計的解釈
各時点は過去推定だが、資産選択や条件設定は研究者の判断を含む。

## 8. 金融工学・運用上の解釈
## 解釈の仕方
最小分散は推定した将来分散を小さくする配分で、リターンや最大DDを直接最適化しません。
相関の変化、特に株式と長期債が同時に下がる状況では、過去の低相関が役立たない場合があります。
上の表で、変動率改善・CAGR変化・最大DD変化が同じ方向かを確認しましょう。
Sharpeが高いだけで「全ての意味で優れている」とは判断しません。



## 9. 限界と追加検証
## 限界と次の改善
この3ETFを現在選んだこと自体に選択バイアスがあります。USDベース、事後調整系列、
実際の税・約定・市場インパクトなしという制約もあります。約21年あってもレジームは少数です。
初回に過去だけで推定するwalk-forwardでも、研究者が過去相場を知っている事実は消せません。
次の候補は銘柄上限、重み変化制約、相関変化の監視、時変短期金利を使うSharpe、
JPY評価、別市場・独立期間での検証です。変更前に評価基準を決め直します。
最終的な採用条件は、自分がどのリスクを抑えたいかを明文化して判断してください。